# YOLOv9c Multi-Class Vehicle & Number Plate Detection Training

This notebook trains a **YOLOv9c** model for **5-class detection**:
- `0` — number_plate
- `1` — bike
- `2` — car
- `3` — bus_microbus
- `4` — large_vehicle

**Boundary Cropping:**
- Class `10` (Boundary) is used as a **cropping guide only** — NOT a detection class
- Images are physically cropped to the boundary region
- All annotations are re-normalized to the cropped coordinate space
- Annotations outside the boundary are discarded
- The model never sees anything outside the boundary

**Features:**
- Per-image boundary cropping with coordinate re-normalization
- Multi-label stratified train/val/test split (70/20/10)
- Optimized for **RTX 3060 12GB VRAM**
- Per-class mAP evaluation and confusion matrix

**Hardware:** RTX 3060 12GB GPU

## 1. Install Required Packages

In [ ]:
# Install PyTorch with CUDA 12.1 support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install other packages
!pip install ultralytics scikit-learn pyyaml matplotlib seaborn

print("✓ All packages installed successfully!")

## 2. Import Libraries and Check GPU

In [ ]:
import torch
import yaml
import shutil
import os
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from collections import defaultdict, Counter
from ultralytics import YOLO

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory: {gpu_mem:.2f} GB")
    if gpu_mem < 10:
        print("⚠ Less than 10GB VRAM — will use batch=4 for safety")
    else:
        print("✓ 12GB VRAM detected — batch=8 will be used")
else:
    print("⚠ No CUDA GPU found! Training will be very slow on CPU.")

## 3. Configuration

**⚠ UPDATE THE PATHS BELOW** to match your dataset location and YOLOv9c model path.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  ⚠⚠⚠  UPDATE THESE PATHS BEFORE RUNNING  ⚠⚠⚠
# ═══════════════════════════════════════════════════════════════════

# Path to the EXPORTED dataset (images/ and labels/ subfolders)
RAW_DATASET_PATH = r"C:\Users\user\Desktop\ITIS\storage\exports"  # ⚠ UPDATE THIS

# Path to YOLOv9c pretrained weights
YOLOV9C_WEIGHTS = r"yolov9c.pt"  # ⚠ UPDATE THIS if different

# Output: where the cropped + split dataset will be created
PREPARED_DATASET_PATH = Path("dataset_prepared")

# Train / Val / Test split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.20
TEST_RATIO  = 0.10

# Detection classes (Boundary class 10 is NOT included — it's only for cropping)
CLASS_NAMES = {
    0: 'number_plate',
    1: 'bike',
    2: 'car',
    3: 'bus_microbus',
    4: 'large_vehicle',
}
NUM_CLASSES = len(CLASS_NAMES)

# Boundary class ID in the exported labels (used for cropping, then discarded)
BOUNDARY_CLASS_ID = 10

# Random seed for reproducibility
RANDOM_SEED = 42

print("✓ Configuration loaded")
print(f"  Raw dataset:  {RAW_DATASET_PATH}")
print(f"  YOLOv9c:      {YOLOV9C_WEIGHTS}")
print(f"  Output:       {PREPARED_DATASET_PATH}")
print(f"  Classes:      {NUM_CLASSES} — {list(CLASS_NAMES.values())}")
print(f"  Boundary ID:  {BOUNDARY_CLASS_ID} (crop guide only, not trained)")
print(f"  Split:        train={TRAIN_RATIO} / val={VAL_RATIO} / test={TEST_RATIO}")

## 4. Scan & Parse Dataset

Read all image–label pairs. For each image:
1. Parse all annotations including the boundary (class 10)
2. Separate boundary from detection annotations
3. Report statistics

In [ ]:
raw_path = Path(RAW_DATASET_PATH)
images_path = raw_path / 'images'
labels_path = raw_path / 'labels'

assert images_path.exists(), f"Images folder not found: {images_path}"
assert labels_path.exists(), f"Labels folder not found: {labels_path}"

# Find all image files
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
all_images = sorted([f for f in images_path.iterdir() if f.suffix.lower() in IMAGE_EXTS])

print(f"Found {len(all_images)} images in {images_path}")

# Parse all labels
# Each entry: (img_path, boundary_ann_or_None, list_of_detection_anns)
parsed_data = []
missing_labels = []
invalid_labels = []
class_counts_raw = Counter()
images_with_boundary = 0
images_without_boundary = 0

for img_file in all_images:
    label_file = labels_path / (img_file.stem + '.txt')
    
    if not label_file.exists():
        missing_labels.append(img_file.name)
        continue
    
    valid = True
    boundary = None       # (cls_id, x, y, w, h)
    detections = []       # [(cls_id, x, y, w, h), ...]
    
    with open(label_file, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                invalid_labels.append((label_file.name, f"line {line_num}: expected 5 values, got {len(parts)}"))
                valid = False
                break
            try:
                cls_id = int(parts[0])
                x, y, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                
                if cls_id == BOUNDARY_CLASS_ID:
                    boundary = (cls_id, x, y, w, h)
                elif cls_id in CLASS_NAMES:
                    detections.append((cls_id, x, y, w, h))
                    class_counts_raw[cls_id] += 1
                # else: skip unknown classes
            except ValueError:
                invalid_labels.append((label_file.name, f"line {line_num}: invalid number format"))
                valid = False
                break
    
    if valid and detections:
        parsed_data.append((img_file, boundary, detections))
        if boundary:
            images_with_boundary += 1
        else:
            images_without_boundary += 1

print(f"\n✓ Parsed {len(parsed_data)} valid image-label pairs")
print(f"  Missing labels: {len(missing_labels)}")
print(f"  Invalid labels: {len(invalid_labels)}")
print(f"  Images WITH boundary:    {images_with_boundary}")
print(f"  Images WITHOUT boundary: {images_without_boundary}")

if invalid_labels:
    for name, reason in invalid_labels[:5]:
        print(f"    ⚠ {name}: {reason}")

print(f"\n📊 Raw class distribution (detection classes only):")
for cls_id in sorted(class_counts_raw.keys()):
    name = CLASS_NAMES.get(cls_id, f'unknown_{cls_id}')
    count = class_counts_raw[cls_id]
    bar = '█' * (count // max(1, max(class_counts_raw.values()) // 40))
    print(f"  [{cls_id}] {name:15s}: {count:6d}  {bar}")

print(f"\n  Total detection annotations: {sum(class_counts_raw.values())}")

## 5. Crop Images to Boundary & Re-normalize Annotations

For each image that has a boundary:
1. **Crop** the image to the boundary bounding box region
2. **Re-normalize** all detection annotation coordinates to the cropped region
3. **Discard** annotations whose center falls outside the boundary
4. **Clip** annotation boxes that partially extend outside the crop

Images without a boundary are kept as-is (no cropping).

The boundary annotation itself is **never** written to the output labels.

In [ ]:
def crop_and_remap(img_path, boundary, detections, min_box_size=0.005):
    """
    Crop image to boundary region, re-normalize annotations.
    
    Args:
        img_path: path to the image file
        boundary: (cls_id, xc, yc, w, h) normalized boundary, or None
        detections: list of (cls_id, xc, yc, w, h) normalized detections
        min_box_size: minimum normalized w or h after remapping (discard tiny boxes)
    
    Returns:
        (cropped_img_array, remapped_detections) or (None, None) on failure
        remapped_detections: list of (cls_id, new_xc, new_yc, new_w, new_h)
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None, None
    
    img_h, img_w = img.shape[:2]
    
    if boundary is None:
        # No boundary — return original image and detections as-is
        return img, detections
    
    # ── Compute crop region in pixel coordinates ──
    _, bxc, byc, bw, bh = boundary
    
    # Boundary box edges (normalized)
    b_left   = bxc - bw / 2
    b_right  = bxc + bw / 2
    b_top    = byc - bh / 2
    b_bottom = byc + bh / 2
    
    # Clamp to [0, 1]
    b_left   = max(0.0, b_left)
    b_top    = max(0.0, b_top)
    b_right  = min(1.0, b_right)
    b_bottom = min(1.0, b_bottom)
    
    # Pixel coordinates
    px_left   = int(b_left * img_w)
    px_top    = int(b_top * img_h)
    px_right  = int(b_right * img_w)
    px_bottom = int(b_bottom * img_h)
    
    # Ensure valid crop
    crop_w = px_right - px_left
    crop_h = px_bottom - px_top
    if crop_w < 10 or crop_h < 10:
        return None, None  # Boundary too small
    
    # Crop the image
    cropped = img[px_top:px_bottom, px_left:px_right]
    
    # ── Re-normalize detections to cropped coordinate space ──
    remapped = []
    for cls_id, axc, ayc, aw, ah in detections:
        # Check if annotation center is inside boundary
        if not (b_left <= axc <= b_right and b_top <= ayc <= b_bottom):
            continue  # Center outside boundary → discard
        
        # Annotation edges (normalized to original image)
        a_left   = axc - aw / 2
        a_top    = ayc - ah / 2
        a_right  = axc + aw / 2
        a_bottom = ayc + ah / 2
        
        # Clip annotation to boundary
        a_left   = max(a_left,   b_left)
        a_top    = max(a_top,    b_top)
        a_right  = min(a_right,  b_right)
        a_bottom = min(a_bottom, b_bottom)
        
        # Re-normalize to cropped coordinate space [0, 1]
        new_left   = (a_left   - b_left) / (b_right - b_left)
        new_top    = (a_top    - b_top)   / (b_bottom - b_top)
        new_right  = (a_right  - b_left) / (b_right - b_left)
        new_bottom = (a_bottom - b_top)   / (b_bottom - b_top)
        
        new_w = new_right - new_left
        new_h = new_bottom - new_top
        new_xc = new_left + new_w / 2
        new_yc = new_top + new_h / 2
        
        # Clamp to [0, 1]
        new_xc = max(0.0, min(1.0, new_xc))
        new_yc = max(0.0, min(1.0, new_yc))
        new_w  = max(0.0, min(1.0, new_w))
        new_h  = max(0.0, min(1.0, new_h))
        
        # Skip tiny boxes
        if new_w < min_box_size or new_h < min_box_size:
            continue
        
        remapped.append((cls_id, new_xc, new_yc, new_w, new_h))
    
    return cropped, remapped


# ── Test on a sample image ──
sample_with_boundary = [d for d in parsed_data if d[1] is not None]
if sample_with_boundary:
    s_img, s_bound, s_dets = sample_with_boundary[0]
    print(f"Sample image: {s_img.name}")
    print(f"  Boundary: xc={s_bound[1]:.3f} yc={s_bound[2]:.3f} w={s_bound[3]:.3f} h={s_bound[4]:.3f}")
    print(f"  Detections before crop: {len(s_dets)}")
    
    test_img, test_dets = crop_and_remap(s_img, s_bound, s_dets)
    if test_img is not None:
        print(f"  Detections after crop:  {len(test_dets)}")
        print(f"  Cropped image size:     {test_img.shape[1]}x{test_img.shape[0]}")
        print(f"\n✓ Crop & remap function verified!")
    else:
        print("  ⚠ Crop failed on sample")
else:
    print("ℹ No images with boundary found — images will be used as-is")

## 6. Multi-Label Stratified Train/Val/Test Split

Since each image can contain **multiple classes** (e.g., a car AND its number plate),
we use a multi-label stratification approach to ensure balanced class distribution
across train, val, and test splits.

**Note:** The split is done BEFORE cropping so we can report statistics. Cropping happens during file copy.

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def multi_label_stratified_split(data, train_r, val_r, test_r, seed=42):
    """
    Stratified split for multi-label data.
    Groups images by their class combination signature, then splits each group
    proportionally to maintain class distribution.
    
    data: list of (img_path, boundary, detections)
    Returns: (train_list, val_list, test_list)
    """
    groups = defaultdict(list)
    for item in data:
        _, _, detections = item
        class_set = frozenset(cls_id for cls_id, *_ in detections)
        groups[class_set].append(item)
    
    train_set, val_set, test_set = [], [], []
    rng = random.Random(seed)
    
    for class_sig, items in groups.items():
        rng.shuffle(items)
        n = len(items)
        n_train = max(1, int(n * train_r))
        n_val = max(1, int(n * val_r)) if n > 2 else 0
        n_test = n - n_train - n_val
        
        if n_test < 0:
            n_test = 0
            n_val = n - n_train
        
        train_set.extend(items[:n_train])
        val_set.extend(items[n_train:n_train + n_val])
        test_set.extend(items[n_train + n_val:])
    
    rng.shuffle(train_set)
    rng.shuffle(val_set)
    rng.shuffle(test_set)
    
    return train_set, val_set, test_set


train_data, val_data, test_data = multi_label_stratified_split(
    parsed_data, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=RANDOM_SEED
)

print(f"✓ Dataset split complete:")
print(f"  Train: {len(train_data):5d} images ({len(train_data)/len(parsed_data)*100:.1f}%)")
print(f"  Val:   {len(val_data):5d} images ({len(val_data)/len(parsed_data)*100:.1f}%)")
print(f"  Test:  {len(test_data):5d} images ({len(test_data)/len(parsed_data)*100:.1f}%)")
print(f"  Total: {len(parsed_data):5d} images")

# Verify class distribution across splits
def count_classes(data_list):
    counts = Counter()
    for _, _, dets in data_list:
        for cls_id, *_ in dets:
            counts[cls_id] += 1
    return counts

train_cls = count_classes(train_data)
val_cls   = count_classes(val_data)
test_cls  = count_classes(test_data)

print(f"\n📊 Per-class distribution across splits (before cropping):")
print(f"  {'Class':<18s} {'Train':>7s} {'Val':>7s} {'Test':>7s} {'Total':>7s}")
print(f"  {'─'*18} {'─'*7} {'─'*7} {'─'*7} {'─'*7}")
for cls_id in sorted(CLASS_NAMES.keys()):
    name = CLASS_NAMES[cls_id]
    t = train_cls.get(cls_id, 0)
    v = val_cls.get(cls_id, 0)
    te = test_cls.get(cls_id, 0)
    total = t + v + te
    print(f"  [{cls_id}] {name:<14s} {t:7d} {v:7d} {te:7d} {total:7d}")

## 7. Crop & Copy Dataset to Split Directories

For each image:
- If it has a boundary → **crop** the image to that region, re-normalize annotations
- If no boundary → copy the image as-is
- Write **only** detection labels (class 10 boundary is never written)

In [ ]:
def process_and_copy_split(split_data, split_name, output_base):
    """
    Crop images to boundary, re-normalize labels, and save to split directory.
    """
    img_dir = output_base / split_name / 'images'
    lbl_dir = output_base / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    
    saved = 0
    cropped_count = 0
    uncropped_count = 0
    skipped = 0
    total_annotations = 0
    
    for img_file, boundary, detections in split_data:
        cropped_img, remapped_dets = crop_and_remap(img_file, boundary, detections)
        
        if cropped_img is None or not remapped_dets:
            skipped += 1
            continue
        
        # Save cropped image
        out_img_path = img_dir / img_file.name
        cv2.imwrite(str(out_img_path), cropped_img)
        
        # Write remapped labels (boundary NOT included)
        out_label_path = lbl_dir / (img_file.stem + '.txt')
        with open(out_label_path, 'w') as f:
            for cls_id, x, y, w, h in remapped_dets:
                f.write(f"{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
        
        saved += 1
        total_annotations += len(remapped_dets)
        if boundary:
            cropped_count += 1
        else:
            uncropped_count += 1
    
    print(f"  {split_name}: {saved} images saved ({cropped_count} cropped, {uncropped_count} uncropped, {skipped} skipped)")
    print(f"         {total_annotations} annotations written")
    return saved


# Clean output directory
if PREPARED_DATASET_PATH.exists():
    print(f"Cleaning existing output: {PREPARED_DATASET_PATH}")
    shutil.rmtree(PREPARED_DATASET_PATH)

print(f"\nProcessing dataset → {PREPARED_DATASET_PATH}...")
print(f"(Cropping to boundary, re-normalizing annotations, discarding outliers)\n")

n_train = process_and_copy_split(train_data, 'train', PREPARED_DATASET_PATH)
n_val   = process_and_copy_split(val_data,   'val',   PREPARED_DATASET_PATH)
n_test  = process_and_copy_split(test_data,  'test',  PREPARED_DATASET_PATH)

print(f"\n✓ Dataset prepared at: {PREPARED_DATASET_PATH.absolute()}")
print(f"  Total: {n_train + n_val + n_test} images ({n_train} train / {n_val} val / {n_test} test)")

## 8. Create data.yaml Configuration

In [ ]:
# Create data.yaml for YOLO training
data_yaml = {
    'path': str(PREPARED_DATASET_PATH.absolute()),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': NUM_CLASSES,
    'names': {int(k): v for k, v in CLASS_NAMES.items()}
}

yaml_path = 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("✓ data.yaml created:\n")
with open(yaml_path, 'r') as f:
    print(f.read())

## 9. Visualize Cropped Samples

Sanity check — view a few **cropped** training images with their re-normalized bounding boxes.
These are exactly what the model will see during training.

In [ ]:
# Color map for classes
COLORS = {
    0: (0, 255, 255),    # number_plate — cyan
    1: (0, 165, 255),    # bike — orange
    2: (0, 255, 0),      # car — green
    3: (255, 0, 0),      # bus_microbus — blue
    4: (0, 0, 255),      # large_vehicle — red
}

# Read cropped images from the prepared dataset
train_img_dir = PREPARED_DATASET_PATH / 'train' / 'images'
train_lbl_dir = PREPARED_DATASET_PATH / 'train' / 'labels'
train_imgs = sorted(list(train_img_dir.glob('*.*')))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
sample_indices = random.sample(range(len(train_imgs)), min(8, len(train_imgs)))

for ax_idx, data_idx in enumerate(sample_indices):
    img_path = train_imgs[data_idx]
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                color = [c/255 for c in COLORS.get(cls_id, (255, 255, 255))]
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor=color, facecolor='none')
                axes[ax_idx//4][ax_idx%4].add_patch(rect)
                axes[ax_idx//4][ax_idx%4].text(x1, y1-5, CLASS_NAMES.get(cls_id, '?'),
                                               color=color, fontsize=8, fontweight='bold',
                                               bbox=dict(boxstyle='round,pad=0.2',
                                                        facecolor='black', alpha=0.7))
    
    axes[ax_idx//4][ax_idx%4].imshow(img)
    axes[ax_idx//4][ax_idx%4].set_title(f"{img_path.name} ({w}x{h})", fontsize=8)
    axes[ax_idx//4][ax_idx%4].axis('off')

plt.suptitle('Cropped Training Images — Model sees ONLY this region', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_cropped_annotations.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved sample_cropped_annotations.png")

## 10. Initialize YOLOv9c Model

Load the pretrained YOLOv9c weights for transfer learning.

In [ ]:
# Verify weights file exists
weights_path = Path(YOLOV9C_WEIGHTS)
assert weights_path.exists(), f"YOLOv9c weights not found: {weights_path.absolute()}"
print(f"YOLOv9c weights: {weights_path.absolute()} ({weights_path.stat().st_size / 1024**2:.1f} MB)")

# Load model
model = YOLO(str(weights_path))
print(f"\n✓ Model loaded successfully!")
print(f"  Architecture: YOLOv9c (compact)")
print(f"  Task: detect")
print(f"  Fine-tuning for {NUM_CLASSES} classes: {list(CLASS_NAMES.values())}")

## 11. Train the Model

Training parameters optimized for **RTX 3060 12GB VRAM**:
- **Model**: YOLOv9c (~50MB, compact but powerful)
- **Batch size**: 8 (safe for 12GB with YOLOv9c + AMP)
- **Image size**: 640 (standard YOLO size)
- **Epochs**: 150 with patience=25 early stopping
- **AMP**: Enabled (FP16 mixed precision — halves memory usage)
- **Augmentations**: mosaic, mixup, HSV jitter — all suited for traffic data

**Expected training time**: ~3-5 hours depending on dataset size.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  TRAINING — RTX 3060 12GB Optimized
# ═══════════════════════════════════════════════════════════════

# Determine safe batch size based on available VRAM
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if gpu_mem_gb >= 11:      # RTX 3060 12GB
        BATCH_SIZE = 8
    elif gpu_mem_gb >= 7:     # RTX 3060 8GB variant or similar
        BATCH_SIZE = 4
    else:
        BATCH_SIZE = 2
else:
    BATCH_SIZE = 2
    gpu_mem_gb = 0

print(f"Using batch size: {BATCH_SIZE} (GPU VRAM: {gpu_mem_gb:.1f} GB)")
print(f"Starting training...\n")

results = model.train(
    data='data.yaml',
    epochs=150,
    imgsz=640,
    batch=BATCH_SIZE,
    device=0,                # Use GPU 0
    workers=4,               # 4 workers for data loading (safe for Windows)
    project='runs/detect',
    name='vehicle_plate_yolov9c',
    
    # Early stopping
    patience=25,
    
    # Mixed precision for memory efficiency
    amp=True,
    
    # Caching — 'ram' loads all images to RAM for speed
    # Set to False if you have < 16GB RAM
    cache='ram',
    
    # Augmentations tuned for traffic/vehicle detection
    mosaic=1.0,              # Mosaic augmentation
    mixup=0.15,              # Mixup augmentation (helpful for multi-class)
    hsv_h=0.015,             # HSV-Hue augmentation
    hsv_s=0.7,               # HSV-Saturation
    hsv_v=0.4,               # HSV-Value
    fliplr=0.5,              # Horizontal flip
    flipud=0.0,              # No vertical flip (vehicles don't flip upside down)
    degrees=0.0,             # No rotation (overhead cam is fixed)
    translate=0.1,           # Translation
    scale=0.5,               # Scale augmentation
    close_mosaic=15,         # Disable mosaic for last 15 epochs for fine-tuning
    
    # Saving
    save=True,
    save_period=25,          # Save checkpoint every 25 epochs
    plots=True,
    
    # Optimizer
    optimizer='auto',        # Let ultralytics pick the best optimizer
    cos_lr=True,             # Cosine LR schedule (smoother convergence)
    lr0=0.01,                # Initial learning rate
    lrf=0.01,                # Final LR factor
    warmup_epochs=3.0,       # Warmup epochs
    
    # Detection head
    box=7.5,                 # Box loss weight
    cls=0.5,                 # Classification loss weight
    dfl=1.5,                 # Distribution focal loss weight
)

print("\n" + "═"*60)
print("✓ Training completed!")
print("═"*60)

## 12. Validate the Model

In [ ]:
# Validate on the validation set
metrics = model.val()

print(f"\n{'═'*50}")
print(f"  Validation Results (val set)")
print(f"{'═'*50}")
print(f"  Overall mAP@50:    {metrics.box.map50:.4f}")
print(f"  Overall mAP@50-95: {metrics.box.map:.4f}")
print(f"  Mean Precision:    {metrics.box.mp:.4f}")
print(f"  Mean Recall:       {metrics.box.mr:.4f}")

print(f"\n  Per-class mAP@50:")
if hasattr(metrics.box, 'maps') and metrics.box.maps is not None:
    for i, cls_name in CLASS_NAMES.items():
        if i < len(metrics.box.maps):
            print(f"    [{i}] {cls_name:15s}: mAP@50-95 = {metrics.box.maps[i]:.4f}")

print(f"\n  Speed:")
if hasattr(metrics, 'speed'):
    for k, v in metrics.speed.items():
        print(f"    {k}: {v:.1f}ms")

## 13. Evaluate on Test Set

In [ ]:
# Load best model and evaluate on test set
best_model_path = Path('runs/detect/vehicle_plate_yolov9c/weights/best.pt')

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"Loaded best model: {best_model_path}")
    
    # Evaluate on test split
    test_metrics = best_model.val(
        data='data.yaml',
        split='test',
        device=0,
        plots=True,
        project='runs/detect',
        name='test_evaluation'
    )
    
    print(f"\n{'═'*50}")
    print(f"  TEST SET Results")
    print(f"{'═'*50}")
    print(f"  mAP@50:    {test_metrics.box.map50:.4f}")
    print(f"  mAP@50-95: {test_metrics.box.map:.4f}")
    print(f"  Precision: {test_metrics.box.mp:.4f}")
    print(f"  Recall:    {test_metrics.box.mr:.4f}")
    
    print(f"\n  Per-class mAP@50-95:")
    if hasattr(test_metrics.box, 'maps') and test_metrics.box.maps is not None:
        for i, cls_name in CLASS_NAMES.items():
            if i < len(test_metrics.box.maps):
                score = test_metrics.box.maps[i]
                bar = '█' * int(score * 30)
                print(f"    [{i}] {cls_name:15s}: {score:.4f}  {bar}")
    
    print(f"\n  Confusion matrix saved to: runs/detect/test_evaluation/")
else:
    print(f"⚠ Best model not found at {best_model_path}")
    print("  Training may still be running or the path is different.")

## 14. Test Predictions on Sample Images

In [ ]:
# Run prediction on sample test images
test_images_dir = PREPARED_DATASET_PATH / 'test' / 'images'

if best_model_path.exists() and test_images_dir.exists():
    best_model = YOLO(str(best_model_path))
    
    test_images = sorted(list(test_images_dir.glob('*.*')))[:8]
    
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    
    for idx, img_path in enumerate(test_images):
        results = best_model.predict(
            source=str(img_path),
            conf=0.25,
            device=0,
            verbose=False
        )
        
        # Get annotated image
        annotated = results[0].plot()
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        ax = axes[idx//4][idx%4]
        ax.imshow(annotated)
        
        # Count detections per class
        det_counts = Counter()
        for box in results[0].boxes:
            cls_id = int(box.cls[0].item())
            det_counts[CLASS_NAMES.get(cls_id, f'cls_{cls_id}')] += 1
        
        det_str = ', '.join([f"{v} {k}" for k, v in det_counts.items()])
        ax.set_title(f"{img_path.name}\n{det_str}", fontsize=8)
        ax.axis('off')
    
    plt.suptitle('Test Predictions — YOLOv9c (boundary-cropped images)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('test_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved test_predictions.png")
else:
    print("⚠ Best model or test images not found. Run training first.")

## 15. Training Summary & Export

In [ ]:
print("═" * 60)
print("  YOLOv9c Training Pipeline — Summary")
print("═" * 60)
print()
print(f"  Model:            YOLOv9c (compact)")
print(f"  Classes:          {NUM_CLASSES} — {list(CLASS_NAMES.values())}")
print(f"  Boundary crop:    Enabled (class 10 used to crop, then discarded)")
print(f"  Dataset:          {len(parsed_data)} total images")
print(f"  Split:            {len(train_data)} train / {len(val_data)} val / {len(test_data)} test")
print(f"  GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print()

run_dir = Path('runs/detect/vehicle_plate_yolov9c')
if run_dir.exists():
    print(f"  Training Results:")
    print(f"    Best weights:   {run_dir / 'weights' / 'best.pt'}")
    print(f"    Last weights:   {run_dir / 'weights' / 'last.pt'}")
    print(f"    Training plots: {run_dir}")
    print(f"    Confusion matrix, PR curve, F1 curve — all in the runs directory")
else:
    print("  ⚠ Training results not found. Run training cell first.")

print()
print("  To use the trained model for inference:")
print("    model = YOLO('runs/detect/vehicle_plate_yolov9c/weights/best.pt')")
print("    results = model.predict(source='image.jpg', conf=0.25)")
print()

# Optional: Export to ONNX
# best_model.export(format='onnx', imgsz=640, half=True)

print("✓ Training pipeline completed successfully!")